In [7]:
import yfinance as yf
import pandas as pd
import numpy as np

hisse_kodu = "PLTR"

print(f"Internete baglaniliyor ve {hisse_kodu} verileri cekiliyor...")
df_raw = yf.download(hisse_kodu, start="2020-01-01", end="2026-04-16")

if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.droplevel(1)

print(f"Veri indirildi. Toplam {len(df_raw)} gun.")
print(df_raw.tail())

Internete baglaniliyor ve PLTR verileri cekiliyor...


[*********************100%***********************]  1 of 1 completed

Veri indirildi. Toplam 1391 gun.
Price            Close        High         Low        Open     Volume
Date                                                                 
2026-04-09  130.490005  139.539993  128.470001  139.395004   92361400
2026-04-10  128.059998  129.199997  122.680000  128.479996  116656800
2026-04-13  132.369995  134.419998  129.149994  130.229996   65772800
2026-04-14  135.699997  138.070007  134.000000  134.427002   52786800
2026-04-15  142.149994  142.580002  134.929993  136.789993   48151900


In [8]:
df = df_raw.copy()

# Trend
df['SMA_20']  = df['Close'].rolling(20).mean()
df['EMA_50']  = df['Close'].ewm(span=50, adjust=False).mean()

# RSI (14)
delta      = df['Close'].diff()
gain       = delta.clip(lower=0)
loss       = -delta.clip(upper=0)
avg_gain   = gain.ewm(com=13, adjust=False).mean()
avg_loss   = loss.ewm(com=13, adjust=False).mean()
rs         = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# MACD
ema12             = df['Close'].ewm(span=12, adjust=False).mean()
ema26             = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

# Bollinger Bantlari
bb_mid               = df['Close'].rolling(20).mean()
bb_std               = df['Close'].rolling(20).std()
df['BB_Upper']       = bb_mid + 2 * bb_std
df['BB_Lower']       = bb_mid - 2 * bb_std
df['BB_Width']       = (df['BB_Upper'] - df['BB_Lower']) / bb_mid
df['Price_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

# ATR (Ortalama Gercek Aralik) - volatilite olcusu
high_low   = df['High'] - df['Low']
high_close = (df['High'] - df['Close'].shift()).abs()
low_close  = (df['Low']  - df['Close'].shift()).abs()
tr         = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['ATR_14'] = tr.ewm(com=13, adjust=False).mean()

# Getiri & Momentum (gecmise bakiyor, leakage yok)
df['Return_1d']  = df['Close'].pct_change(1)
df['Return_5d']  = df['Close'].pct_change(5)
df['Return_20d'] = df['Close'].pct_change(20)

# Hacim indikatoru
df['Volume_SMA20'] = df['Volume'].rolling(20).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA20']

df.dropna(inplace=True)
print(f"Indikatorler hazir. Temiz satir: {len(df)}")
print(df[['Close','RSI_14','MACD','ATR_14','Return_5d','Volume_Ratio']].tail())

Indikatorler hazir. Temiz satir: 1371
Price            Close     RSI_14      MACD    ATR_14  Return_5d  Volume_Ratio
Date                                                                          
2026-04-09  130.490005  35.565231 -2.274443  7.954552  -0.109222      2.109973
2026-04-10  128.059998  34.126154 -3.436757  7.944227  -0.137411      2.488090
2026-04-13  132.369995  38.852154 -3.964418  7.831068  -0.105185      1.368793
2026-04-14  135.699997  42.296711 -4.067009  7.678850  -0.095755      1.076269
2026-04-15  142.149994  48.364124 -3.586508  7.676790   0.009875      0.970848


In [9]:
# Hedef: yarin kapanis > bugun kapanis mi?
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# KRITIK: Son gun canli tahmin icin ayrilir, modele verilmez
bugunun_verisi = df.iloc[[-1]].copy()
df_model       = df.iloc[:-1].copy()

features = [
    'SMA_20', 'EMA_50',
    'RSI_14',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Width', 'Price_Position',
    'ATR_14',
    'Return_1d', 'Return_5d', 'Return_20d',
    'Volume_Ratio'
]

X = df_model[features]
y = df_model['Target']

print(f"Toplam ornek: {len(X)} | Ozellik sayisi: {len(features)}")
print(f"Alim (1): {y.sum()} | Satim (0): {(y==0).sum()} | Denge: {y.mean():.2%}")

Toplam ornek: 1370 | Ozellik sayisi: 13
Alim (1): 696 | Satim (0): 674 | Denge: 50.80%


In [10]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# TimeSeriesSplit: 5 farkli donemde walk-forward test
# Klasik train_test_split yerine bu kullan! Borsada gecmisle gelecegi tahmin edersin.
tscv = TimeSeriesSplit(n_splits=5)

modeller = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=20,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=20,
        subsample=0.8,
        random_state=42
    )
}

sonuclar = {}

for isim, model in modeller.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    
    fold_skorlari = []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        skor = accuracy_score(y_te, pipe.predict(X_te))
        fold_skorlari.append(skor)
    
    ort = np.mean(fold_skorlari)
    std = np.std(fold_skorlari)
    sonuclar[isim] = {'pipe': pipe, 'skor': ort, 'std': std, 'folds': fold_skorlari}
    
    print(f"\n{'='*45}")
    print(f"  {isim}")
    print(f"  Ortalama Dogruluk : %{ort*100:.2f} +/- {std*100:.2f}")
    print(f"  Fold Skorlari     : {[round(s*100,1) for s in fold_skorlari]}")

en_iyi_isim = max(sonuclar, key=lambda k: sonuclar[k]['skor'])
en_iyi_pipe = sonuclar[en_iyi_isim]['pipe']
print(f"\nKazanan Model: {en_iyi_isim} (%{sonuclar[en_iyi_isim]['skor']*100:.2f})")


  Random Forest
  Ortalama Dogruluk : %47.46 +/- 3.44
  Fold Skorlari     : [43.9, 48.7, 47.8, 43.9, 53.1]

  Gradient Boosting
  Ortalama Dogruluk : %48.51 +/- 1.38
  Fold Skorlari     : [48.2, 48.7, 50.0, 46.1, 49.6]

Kazanan Model: Gradient Boosting (%48.51)


In [11]:
# Final egitim: Tum gecmis veriyle
en_iyi_pipe.fit(X, y)

# Ozellik onemi
try:
    clf  = en_iyi_pipe.named_steps['clf']
    onem = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)
    print("Ozellik Onem Siralama:")
    print(onem.round(4).to_string())
except Exception as e:
    print(f"Hesaplanamadi: {e}")

# Canli Tahmin Raporu
def analist_raporu(bugun, pipe, features, en_iyi_isim):
    girdi   = bugun[features]
    tahmin  = pipe.predict(girdi)[0]
    ihtimal = pipe.predict_proba(girdi)[0]
    
    kapanis = bugun['Close'].values[0]
    rsi     = bugun['RSI_14'].values[0]
    macd    = bugun['MACD'].values[0]
    atr     = bugun['ATR_14'].values[0]
    r5      = bugun['Return_5d'].values[0]
    vr      = bugun['Volume_Ratio'].values[0]
    
    print("\n" + "=" * 45)
    print("   MIDAS OTONOM ANALIST v2.0")
    print("=" * 45)
    print(f"  Son Kapanis : ${kapanis:.2f}")
    print(f"  Model       : {en_iyi_isim}")
    print()
    
    if tahmin == 1:
        print(f"  YARIN: YUKSELIS")
        print(f"  Guven : %{ihtimal[1]*100:.1f}")
    else:
        print(f"  YARIN: DUSUS")
        print(f"  Guven : %{ihtimal[0]*100:.1f}")
    
    print("-" * 45)
    rsi_yorum = "Asiri satim — tepki alimi gelebilir" if rsi < 30 else ("Asiri alim — kar satisi riski" if rsi > 70 else "Notr")
    print(f"  RSI (14)   : {rsi:.1f}  -> {rsi_yorum}")
    print(f"  MACD       : {macd:.2f} -> {'Yukselis' if macd > 0 else 'Dusus'} momentumu")
    print(f"  ATR (14)   : {atr:.2f} (gunluk oynaklık tahmini)")
    print(f"  5g Getiri  : {r5*100:.1f}%")
    vr_yorum = "Guclu hacim" if vr > 1.2 else ("Normal" if vr > 0.8 else "Zayif hacim")
    print(f"  Hacim x    : {vr:.2f} -> {vr_yorum}")
    print("=" * 45)
    print("  Bu rapor yatirim tavsiyesi degildir.")
    print("=" * 45)

analist_raporu(bugunun_verisi, en_iyi_pipe, features, en_iyi_isim)

Ozellik Onem Siralama:
Volume_Ratio      0.1227
BB_Width          0.1143
Return_5d         0.0995
Return_1d         0.0839
RSI_14            0.0775
Price_Position    0.0743
Return_20d        0.0736
MACD_Hist         0.0697
MACD_Signal       0.0627
EMA_50            0.0625
MACD              0.0617
SMA_20            0.0509
ATR_14            0.0466

   MIDAS OTONOM ANALIST v2.0
  Son Kapanis : $142.15
  Model       : Gradient Boosting

  YARIN: YUKSELIS
  Guven : %61.8
---------------------------------------------
  RSI (14)   : 48.4  -> Notr
  MACD       : -3.59 -> Dusus momentumu
  ATR (14)   : 7.68 (gunluk oynaklık tahmini)
  5g Getiri  : 1.0%
  Hacim x    : 0.97 -> Normal
  Bu rapor yatirim tavsiyesi degildir.
